# Week 25, Day 4: Rotational dynamics: fitting dx/dt = Mx and jPCA

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w025/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

This lesson uses a dataset from the Engram site. Run this cell first: it downloads the file once and defines `dataset()`.

In [ ]:
import os, urllib.request
def dataset(name):
    """Downloads an Engram dataset the first time it is used and returns its local path."""
    if not os.path.exists(name):
        urllib.request.urlretrieve("https://engram-neuroai.vercel.app/data/" + name, name)
    return name

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
data = np.load(dataset("churchland2012_reach.npz"))
rates, times = data["rates"].astype(float), data["times"]
C, T, N = rates.shape
Xn = rates / (rates.max(axis=(0, 1)) - rates.min(axis=(0, 1)) + 5)
X = Xn - Xn.mean(axis=0, keepdims=True)
win = (times >= -50) & (times <= 150)
Xw = X[:, win, :]
D = Xw.reshape(-1, N); mu = D.mean(axis=0)
Vt = np.linalg.svd(D - mu, full_matrices=False)[2]
Z = (Xw - mu) @ Vt[:6].T                       # conditions x 21 times x 6 components
dt = 0.01                                     # 10 ms bins, in seconds
print("Z:", Z.shape)

### Exercise: An unconstrained linear fit

Write `pairs(Zc)` returning the states (all but the last time point) and finite-difference derivatives of a set of conditions, stacked into rows, and `fit_full(Zm, dZ)`, which returns M from the least-squares solution of Zm Mᵀ = dZ. Fit all conditions for `M_full` and `r2_full`, then fit even conditions and test on odd ones for `r2_full_test`.

In [ ]:
def pairs(Zc):
    return Zc[:, :-1].reshape(-1, 6), ...

def r2(Zm, dZ, M):
    return 1 - np.sum((dZ - Zm @ M.T) ** 2) / np.sum((dZ - dZ.mean(0)) ** 2)

def fit_full(Zm, dZ):
    return ...

Zm, dZ = pairs(Z)
M_full = fit_full(Zm, dZ); r2_full = r2(Zm, dZ, M_full)
even, odd = np.arange(0, C, 2), np.arange(1, C, 2)
r2_full_test = r2(*pairs(Z[odd]), fit_full(*pairs(Z[even])))
print(f"R2 {r2_full:.3f} in sample, {r2_full_test:.3f} on held-out conditions")
print("eigenvalues of M (per second):", np.round(np.linalg.eigvals(M_full), 2))

In [ ]:
import numpy as np
def test_pairs():
    "Zm and dZ stack every condition's states and finite-difference derivatives"
    assert Zm.shape == (C * 20, 6) and np.allclose(dZ, (np.diff(Z, axis=1) / dt).reshape(-1, 6))

def test_fit():
    "M_full is the least-squares solution and r2_full its R squared"
    Mref = np.linalg.lstsq(Zm, dZ, rcond=None)[0].T
    assert np.allclose(M_full, Mref) and abs(r2_full - r2(Zm, dZ, Mref)) < 1e-12

def test_heldout():
    "the held-out R squared on odd conditions is a little below the in-sample value"
    assert 0.5 < r2_full_test < r2_full

test_pairs()
test_fit()
test_heldout()
print("All checks passed")

### Exercise: Fitting a rotation

Complete `fit_skew(Zm, dZ)`: build the 15 basis matrices, form the design matrix whose columns are the flattened predictions Zm Eᵀ for each basis matrix, solve for the coefficients by least squares and return their weighted sum. Fit all conditions for `M_skew` and `r2_skew`, and even-to-odd for `r2_skew_test`.

In [ ]:
basis = []
for i in range(6):
    for j in range(i + 1, 6):
        E = np.zeros((6, 6)); E[i, j], E[j, i] = 1, -1
        basis.append(E)

def fit_skew(Zm, dZ):
    G = np.stack([(Zm @ E.T).ravel() for E in basis], axis=1)
    h = ...
    return sum(c * E for c, E in zip(h, basis))

M_skew = fit_skew(Zm, dZ); r2_skew = r2(Zm, dZ, M_skew)
r2_skew_test = ...
print(f"rotation only: R2 {r2_skew:.3f} in sample, {r2_skew_test:.3f} held out;  ratio to the free fit {r2_skew / r2_full:.2f}")

In [ ]:
import numpy as np
def test_skew():
    "M_skew is skew-symmetric"
    assert np.allclose(M_skew, -M_skew.T)

def test_best():
    "M_skew is the best skew-symmetric fit: random skew perturbations never improve it"
    r = np.random.default_rng(3)
    for _ in range(20):
        P = r.normal(size=(6, 6)) * 0.5; P = P - P.T
        assert r2(Zm, dZ, M_skew + P) <= r2_skew + 1e-12

def test_values():
    "the rotation explains less than the free fit but more than half as much, also on held-out conditions"
    assert 0.5 * r2_full < r2_skew < r2_full and 0.5 * r2_full_test < r2_skew_test < r2_full_test

test_skew()
test_best()
test_values()
print("All checks passed")

### Exercise: The plane of fastest rotation

Find the eigenvalue of M_skew with the largest imaginary part, form `P` by orthonormalising (QR) the real and imaginary parts of its eigenvector, and project Z onto it to get `J`. Store the frequency in Hz in `freq_hz`. For each condition, take the sign of its mean angular velocity in the plane (the average of J₁ dJ₂ − J₂ dJ₁), and store the fraction of conditions that share the majority sign in `frac_same`.

In [ ]:
ev, V = np.linalg.eig(M_skew)
i = np.argmax(ev.imag)
P = ...
J = Z @ P                                       # conditions x 21 x 2
freq_hz = ...
dJ = np.diff(J, axis=1)
spin = (J[:, :-1, 0] * dJ[:, :, 1] - J[:, :-1, 1] * dJ[:, :, 0]).mean(axis=1)
frac_same = ...
print(f"rotation at {freq_hz:.2f} Hz;  {100 * frac_same:.0f}% of conditions turn the same way")
fig, ax = plt.subplots(figsize=(5, 5))
for c in range(C):
    ax.plot(J[c, :, 0], J[c, :, 1], lw=0.7); ax.plot(J[c, -1, 0], J[c, -1, 1], "k.", ms=3)
ax.set_aspect("equal"); ax.set_xlabel("jPC1"); ax.set_ylabel("jPC2"); plt.show()

In [ ]:
import numpy as np
def test_plane():
    "P has two orthonormal columns spanning the real and imaginary parts of the top eigenvector"
    ev, V = np.linalg.eig(M_skew); i = np.argmax(ev.imag)
    B = np.stack([V[:, i].real, V[:, i].imag], 1)
    assert np.allclose(P.T @ P, np.eye(2), atol=1e-9) and np.allclose(P @ (P.T @ B), B, atol=1e-9)

def test_freq():
    "freq_hz is the top rotation frequency, between 0.5 and 3 Hz"
    assert abs(freq_hz - np.max(np.linalg.eigvals(M_skew).imag) / (2 * np.pi)) < 1e-9 and 0.5 < freq_hz < 3

def test_consistent():
    "more than 90% of conditions rotate in the majority direction"
    assert frac_same > 0.9

test_plane()
test_freq()
test_consistent()
print("All checks passed")